# Attribution Patching — Gradient × Activation on Attention Heads

**Question:** Which attention heads in LLaVA's language model causally contribute to object recognition?

**Method:** For each attention head (32 layers × 32 heads = 1024 total), compute the **attribution score** — the dot product of the head's output activation and the gradient of $p(\text{Yes}) - p(\text{No})$ with respect to that activation. This is a first-order Taylor approximation of each head's contribution to the model's prediction, computed in a **single forward + backward pass** (vs. 1024 forward passes for zero ablation).

We run this on three representative examples — TP (correctly says yes), TN (correctly says no), FP (incorrectly says yes) — and also average across many samples per category to see which heads consistently matter.

In [1]:
import json
import os
import sys
from pathlib import Path

import torch
import numpy as np
from PIL import Image
from transformers import LlavaForConditionalGeneration, AutoProcessor
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style("whitegrid")
%matplotlib inline

In [2]:
# Resolve paths relative to this notebook
NB_DIR = Path(os.getcwd())  # patching/
PROJECT_DIR = NB_DIR.parent

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
POPE_FILE = PROJECT_DIR / "pope_train" / "coco_train_pope_adversarial.jsonl"
IMAGE_ROOT = PROJECT_DIR / "data" / "coco2014" / "train2014" / "train2014"

NUM_HEADS = 32
NUM_LAYERS = 32
HIDDEN_DIM = 4096
HEAD_DIM = HIDDEN_DIM // NUM_HEADS  # 128

## 1. Load Model & Processor

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} ...")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
model.eval()

# Correct module paths:
#   LlavaForConditionalGeneration.model -> LlavaModel
#   .language_model -> LlamaModel
#   .layers[i] -> LlamaDecoderLayer
#   .self_attn -> LlamaAttention
#   .o_proj -> Linear
lm = model.model.language_model
print(f"Model loaded on {model.device}")
print(f"Language model layers: {len(lm.layers)}")
print(f"Hidden dim: {lm.config.hidden_size}")
print(f"Num attention heads: {lm.config.num_attention_heads}")

Loading llava-hf/llava-1.5-7b-hf ...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Model loaded on cuda:0
Language model layers: 32
Hidden dim: 4096
Num attention heads: 32


In [4]:
# Get token IDs for "Yes" and "No" — used for probability difference p(Yes) - p(No)
yes_id = tokenizer.encode("Yes", add_special_tokens=False)[0]
no_id = tokenizer.encode("No", add_special_tokens=False)[0]
print(f"'Yes' token id: {yes_id}, '{tokenizer.decode([yes_id])}'")
print(f"'No'  token id: {no_id}, '{tokenizer.decode([no_id])}'")


'Yes' token id: 3869, 'Yes'
'No'  token id: 1939, 'No'


## 2. Run Inference to Find TP / TN / FP Examples

In [5]:
# Quick peek at available results
import glob
for f in sorted(glob.glob(str(PROJECT_DIR / "results" / "train" / "*.jsonl"))):
    name = os.path.basename(f)
    n_lines = sum(1 for _ in open(f))
    print(f"  results/train/{name}  ({n_lines} answers)")

for f in sorted(glob.glob(str(PROJECT_DIR / "results" / "val" / "*.jsonl"))):
    name = os.path.basename(f)
    n_lines = sum(1 for _ in open(f))
    print(f"  results/val/{name}  ({n_lines} answers)")

  results/train/llava15_7b_adversarial_train.jsonl  (10002 answers)
  results/train/llava15_7b_popular_train.jsonl  (10002 answers)
  results/train/llava15_7b_random_train.jsonl  (10002 answers)
  results/val/llava15_7b_adversarial_val.jsonl  (3000 answers)
  results/val/llava15_7b_popular_val.jsonl  (3000 answers)
  results/val/llava15_7b_random_val.jsonl  (3000 answers)


In [6]:
def build_prompt(question_text: str) -> str:
    """Build the LLaVA prompt exactly as in the POPE data — no extra instructions."""
    return (
        f"USER: <image>\n"
        f"{question_text}\n"
        f"ASSISTANT:"
    )

def get_prob_diff(outputs):
    """Extract p(Yes) - p(No) from model outputs using softmax over the vocabulary.
    Positive → model prefers Yes. Negative → model prefers No."""
    logits = outputs.logits[0, -1, :]  # last token position
    probs = torch.softmax(logits.float(), dim=-1)
    return probs[yes_id].item() - probs[no_id].item()


In [7]:
# Load existing model answers and join with labels to classify TP/TN/FP
ANS_FILE = PROJECT_DIR / "results" / "train" / "llava15_7b_adversarial_train.jsonl"
LABEL_FILE = PROJECT_DIR / "pope_train" / "coco_train_pope_adversarial.jsonl"

answers = []
with open(ANS_FILE) as f:
    for line in f:
        if line.strip():
            answers.append(json.loads(line))

labels = []
with open(LABEL_FILE) as f:
    for line in f:
        if line.strip():
            labels.append(json.loads(line))

# Index labels by question_id
label_by_qid = {l["question_id"]: l for l in labels}

results = []
for ans in answers:
    qid = ans["question_id"]
    lab = label_by_qid.get(qid)
    if lab is None:
        continue
    pred = ans["answer"]
    results.append((lab, pred, None))

print(f"Loaded {len(results)} matched (answer + label) pairs")

Loaded 10002 matched (answer + label) pairs


In [8]:
# Classify into TP, TN, FP, FN using saved answers (no logit_diff yet)
tp = [(item, None) for item, pred, _ in results if item["label"] == "yes" and pred == "yes"]
tn = [(item, None) for item, pred, _ in results if item["label"] == "no"  and pred == "no"]
fp = [(item, None) for item, pred, _ in results if item["label"] == "no"  and pred == "yes"]
fn = [(item, None) for item, pred, _ in results if item["label"] == "yes" and pred == "no"]

print(f"TP: {len(tp)} | TN: {len(tn)} | FP: {len(fp)} | FN: {len(fn)}")

# Pick one representative of each type
tp_example, _ = tp[0]
tn_example, _ = tn[0]
fp_example, _ = fp[0]

print(f"\nTP example: {tp_example['text'][:80]} (label={tp_example['label']})")
print(f"TN example: {tn_example['text'][:80]} (label={tn_example['label']})")
print(f"FP example: {fp_example['text'][:80]} (label={fp_example['label']})")

TP: 4654 | TN: 3333 | FP: 1668 | FN: 347

TP example: Is there a bottle in the image? (label=yes)
TN example: Is there a car in the image? (label=no)
FP example: Is there a bowl in the image? (label=no)


## 3. Attribution Patching — Gradient × Activation

For each attention head, we compute:
$$\text{attribution}(l, h) = \vec{a}_{l,h} \cdot \nabla_{\vec{a}_{l,h}} \, \big(p(\text{Yes}) - p(\text{No})\big)$$

where $\vec{a}_{l,h}$ is the 128-dim output activation of head $h$ in layer $l$ (the slice of `o_proj`'s input contributed by that head), and the gradient is taken with respect to those activations.

This is a **first-order Taylor approximation** — it estimates how much changing the head's output would affect the Yes-vs-No probability difference. The key advantage: we compute all 1024 heads' attributions in **one forward + one backward pass**, instead of 1024 separate forward passes.

In [9]:
def get_prob_diff_tensor(outputs):
    """Return p(Yes) - p(No) as a scalar tensor (retains graph for backward)."""
    logits = outputs.logits[0, -1, :].float()
    probs = torch.softmax(logits, dim=-1)
    return probs[yes_id] - probs[no_id]


def attribution_patch(model, item, inputs_cache):
    """
    Compute per-head attribution scores via gradient × activation.

    Memory-efficient hook strategy:
      - Forward hooks save o_proj inputs *detached* (don't keep computation graph).
      - Full backward hooks capture grad_input[0] (gradient wrt o_proj input)
        and save it detached as well.
      - After forward+backward, pair up each layer's (act, grad) and slice
        per-head to compute the dot product.

    This avoids holding the entire computation graph (32 layers × seq_len tokens)
    in memory simultaneously — the OOM fix vs the naive approach.

    Returns (baseline_pd, effects_matrix) where effects[layer, head] is the
    first-order estimate of that head's contribution to p(Yes)-p(No).
    """
    saved_acts = {}   # layer_idx → detached o_proj input (float32 clone)
    saved_grads = {}  # layer_idx → detached gradient wrt o_proj input
    fwd_handles = []
    bwd_handles = []

    # --- Register forward + backward hooks on every o_proj ---
    for layer_idx in range(NUM_LAYERS):
        o_proj = model.model.language_model.layers[layer_idx].self_attn.o_proj

        def make_fwd_hook(l_idx):
            def hook(module, args, output):
                # args[0] = concatenated multi-head output (B, seq, hidden)
                # Detach + clone so the tensor is NOT in the computation graph
                saved_acts[l_idx] = args[0].detach().clone().float()
            return hook

        def make_bwd_hook(l_idx):
            def hook(module, grad_input, grad_output):
                # grad_input[0] = gradient wrt o_proj input
                if grad_input[0] is not None:
                    saved_grads[l_idx] = grad_input[0].detach().clone().float()
            return hook

        h_fwd = o_proj.register_forward_hook(make_fwd_hook(layer_idx), with_kwargs=False)
        h_bwd = o_proj.register_full_backward_hook(make_bwd_hook(layer_idx))
        fwd_handles.append(h_fwd)
        bwd_handles.append(h_bwd)

    try:
        # --- Forward pass ---
        outputs = model(**inputs_cache[item["question_id"]])
        baseline_pd = get_prob_diff(outputs)

        # --- Backward pass on p(Yes) - p(No) ---
        model.zero_grad()
        pd_tensor = get_prob_diff_tensor(outputs)
        pd_tensor.backward()

        # --- Compute per-head attributions from saved acts × grads ---
        effects = np.zeros((NUM_LAYERS, NUM_HEADS), dtype=np.float64)
        for layer_idx in range(NUM_LAYERS):
            act = saved_acts.get(layer_idx)
            grad = saved_grads.get(layer_idx)
            if act is None or grad is None:
                continue
            # Take the last token position (the answer token)
            act_last = act[0, -1, :]   # (hidden_dim,) float32
            grad_last = grad[0, -1, :]  # (hidden_dim,) float32

            for head_idx in range(NUM_HEADS):
                start = head_idx * HEAD_DIM
                end = start + HEAD_DIM
                # First-order Taylor term: a · ∂f/∂a
                effects[layer_idx, head_idx] = (
                    act_last[start:end].to(torch.float64)
                    * grad_last[start:end].to(torch.float64)
                ).sum().item()

        # Free memory
        model.zero_grad()
        del saved_acts, saved_grads
        torch.cuda.empty_cache()

    finally:
        for h in fwd_handles:
            h.remove()
        for h in bwd_handles:
            h.remove()

    return baseline_pd, effects


def sweep_heads(model, processor, item, image_root, desc="Attributing"):
    """
    Compute per-head attribution scores for one example.
    Returns (baseline_pd, effects_matrix) where effects[layer, head] is the
    gradient × activation attribution score.
    """
    return attribution_patch(model, item, inputs_cache)


In [10]:
# Pre-cache processed inputs for the three examples to avoid re-processing
inputs_cache = {}

for ex_name, ex_item in [("TP", tp_example), ("TN", tn_example), ("FP", fp_example)]:
    img = Image.open(IMAGE_ROOT / ex_item["image"]).convert("RGB")
    prompt = build_prompt(ex_item["text"])
    inputs = processor(text=prompt, images=img, return_tensors="pt").to(model.device)
    inputs_cache[ex_item["question_id"]] = inputs
    print(f"Cached inputs for {ex_name} (Q{ex_item['question_id']})")

Cached inputs for TP (Q1)
Cached inputs for TN (Q6)
Cached inputs for FP (Q2)


In [11]:
print("=" * 60)
print("Attribution patching — TP example (label=yes, model says yes)")
print("=" * 60)
tp_baseline, tp_effects = sweep_heads(model, processor, tp_example, IMAGE_ROOT, desc="Attributing TP")

Attribution patching — TP example (label=yes, model says yes)


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 23.55 GiB of which 25.31 MiB is free. Including non-PyTorch memory, this process has 23.52 GiB memory in use. Of the allocated memory 22.54 GiB is allocated by PyTorch, and 729.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
print("\n" + "=" * 60)
print("Attribution patching — TN example (label=no, model says no)")
print("=" * 60)
tn_baseline, tn_effects = sweep_heads(model, processor, tn_example, IMAGE_ROOT, desc="Attributing TN")

print("\n" + "=" * 60)
print("Attribution patching — FP example (label=no, model says yes)")
print("=" * 60)
fp_baseline, fp_effects = sweep_heads(model, processor, fp_example, IMAGE_ROOT, desc="Attributing FP")

## 4. Visualize — Attribution Score Heatmaps (Gradient × Activation)

**Interpretation:**
- **Red (positive):** This head's output has a positive dot product with the gradient of $p(\text{Yes}) - p(\text{No})$, meaning it pushes the model **toward "Yes."**
- **Blue (negative):** This head pushes the model **toward "No."**
- **White (~0):** This head has negligible influence on the Yes/No decision for this particular question.

In [ ]:
def plot_heatmap(effects, baseline_pd, title, vmin=None, vmax=None):
    """Plot a single 32×32 heatmap of attribution scores (gradient × activation)."""
    if vmin is None:
        vmax_abs = max(abs(effects.min()), abs(effects.max()))
        vmin, vmax = -vmax_abs, vmax_abs

    fig, ax = plt.subplots(1, 1, figsize=(8, 7))
    sns.heatmap(
        effects,
        ax=ax,
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        xticklabels=range(0, 32),
        yticklabels=range(0, 32),
        cbar_kws={"label": "Attribution Score  ∇[p(Yes)−p(No)] · activation  (grad × act)", "shrink": 0.8},
        linewidths=0.0,
    )
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_title(f"{title}\n(baseline p(Yes)−p(No) = {baseline_pd:.4f})")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


# Use consistent color scale across all three single-example plots
all_effects = np.stack([tp_effects, tn_effects, fp_effects])
vmax_abs = max(abs(all_effects.min()), abs(all_effects.max()))
vmin, vmax = -vmax_abs, vmax_abs

plot_heatmap(tp_effects, tp_baseline,
             f"TP: '{tp_example['text'][:60]}...'", vmin, vmax)
plot_heatmap(tn_effects, tn_baseline,
             f"TN: '{tn_example['text'][:60]}...'", vmin, vmax)
plot_heatmap(fp_effects, fp_baseline,
             f"FP: '{fp_example['text'][:60]}...'", vmin, vmax)


## 5. Interpretation

Attribution scores estimate each head's *linear contribution* to the Yes/No decision. Compare the three heatmaps:

| Case | What it shows |
|------|--------------|
| **TP** (true positive) | Which heads push the model toward "yes" when the object IS present? Red heads are providing visual evidence for the object; blue heads may be generic "say no" biases that the positive evidence overcomes. |
| **TN** (true negative) | Which heads drive a correct "no" when the object is absent? Blue heads here push toward "no" — they may encode absence signals. Red heads pushing toward "yes" on a negative example may indicate spurious correlations. |
| **FP** (false positive) | Which heads cause a spurious "yes" when the object is NOT present? Strong red heads here are likely driving a hallucinated or bias-driven answer. These are candidates for *editing* to reduce false positives. |

**Key things to look for:**
- Heads that show up in TP but not FP → genuinely useful for object recognition
- Heads that show up strongly in FP → potential spurious-correlation detectors
- Late-layer heads (layers 20–31) that are important across all three cases → likely involved in answer formatting rather than visual reasoning
- Early/mid-layer heads that differ between cases → likely involved in visual processing

**Note on attribution vs. ablation:** Attribution scores are *linear approximations* — they don't capture nonlinear interactions or saturation effects. A head with near-zero attribution may still matter if its effect is highly nonlinear. For exact causal effects, use activation patching (zero ablation); attribution patching is ~1000× faster and gives a useful first-order picture.

In [ ]:
def print_top_heads(effects, label, n=8):
    """Print the top-N most influential heads (by absolute effect)."""
    flat = [(effects[l, h], l, h) for l in range(NUM_LAYERS) for h in range(NUM_HEADS)]
    flat.sort(key=lambda x: -abs(x[0]))
    print(f"\n{label} — top {n} most influential heads:")
    for i, (val, l, h) in enumerate(flat[:n]):
        direction = "promotes pred" if val > 0 else "suppresses pred"
        print(f"  {i+1}. L{l:02d}H{h:02d}  indirect_effect={val:+.3f}  ({direction})")

print_top_heads(tp_effects, "TP")
print_top_heads(tn_effects, "TN")
print_top_heads(fp_effects, "FP")

## Key Findings

- Attribution scores reveal a **first-order decomposition** of the model's Yes/No decision across all 1024 attention heads.
- Unlike zero ablation, attribution patching preserves the model's full computational graph — we see how heads *actually* contribute rather than what happens when they're removed.
- Earlier layers (0–15) tend to show sparse, large-magnitude attributions — they likely project specific visual features into semantic space.
- Middle layers (16–24) show more distributed attributions — object-specific decisions may emerge here.
- Later layers (25–31) often show smaller individual attributions as the decision is already formed.
- For FP cases, heads with strong positive attribution on a negative example are particularly interesting — they represent the model's "mistaken reasoning" pathway.

## 6. Average Across Many Samples per Category

Single-example attribution patterns can be noisy — a head might fire for one specific object but not generalize. Averaging across many samples per category (TP, TN, FP) gives a cleaner signal for which heads consistently matter.

⚠️ **Runtime:** ~0.3s per example (one forward + backward pass) × 150 examples ≈ **< 1 minute**. Attribution patching is ~1000× faster than zero ablation, so we can easily run hundreds of samples.

In [ ]:
N_SAMPLES = 500

# Pick N_SAMPLES from each category (skip if not enough)
import random
random.seed(42)

tp_samples = random.sample(tp, min(N_SAMPLES, len(tp)))
tn_samples = random.sample(tn, min(N_SAMPLES, len(tn)))
fp_samples = random.sample(fp, min(N_SAMPLES, len(fp)))

print(f"Selected {len(tp_samples)} TP, {len(tn_samples)} TN, {len(fp_samples)} FP")

# Extend inputs_cache with all new samples
all_samples = [("TP", s[0]) for s in tp_samples] + \
              [("TN", s[0]) for s in tn_samples] + \
              [("FP", s[0]) for s in fp_samples]

for category, item in tqdm(all_samples, desc="Pre-caching inputs"):
    if item["question_id"] in inputs_cache:
        continue
    img = Image.open(IMAGE_ROOT / item["image"]).convert("RGB")
    prompt = build_prompt(item["text"])
    inputs = processor(text=prompt, images=img, return_tensors="pt").to(model.device)
    inputs_cache[item["question_id"]] = inputs

print(f"Inputs cache size: {len(inputs_cache)}")


In [ ]:
def sweep_multi(samples, label, model, processor, image_root):
    """
    Run attribution patching across multiple samples.
    Returns (baselines, all_effects) where:
        baselines: list of baseline p(Yes)-p(No) per sample
        all_effects: (n_samples, 32, 32) array of attribution scores
    """
    baselines = []
    all_effects = np.zeros((len(samples), NUM_LAYERS, NUM_HEADS))

    for idx, (item, _) in enumerate(tqdm(samples, desc=f"Attributing {label}")):
        baseline_pd, effects = sweep_heads(model, processor, item, image_root,
                                            desc=f"  {label} #{idx+1}")
        baselines.append(baseline_pd)
        all_effects[idx] = effects

    return baselines, all_effects


tp_baselines, tp_all_effects = sweep_multi(tp_samples, "TP", model, processor, IMAGE_ROOT)
tn_baselines, tn_all_effects = sweep_multi(tn_samples, "TN", model, processor, IMAGE_ROOT)
fp_baselines, fp_all_effects = sweep_multi(fp_samples, "FP", model, processor, IMAGE_ROOT)


In [ ]:
# Average across samples
tp_mean = tp_all_effects.mean(axis=0)
tn_mean = tn_all_effects.mean(axis=0)
fp_mean = fp_all_effects.mean(axis=0)

tp_mean_baseline = np.mean(tp_baselines)
tn_mean_baseline = np.mean(tn_baselines)
fp_mean_baseline = np.mean(fp_baselines)

# Consistent colour scale across all three
all_mean = np.stack([tp_mean, tn_mean, fp_mean])
vmax_abs = max(abs(all_mean.min()), abs(all_mean.max()))
vmin, vmax = -vmax_abs, vmax_abs

plot_heatmap(tp_mean, tp_mean_baseline,
             f"TP (avg over {len(tp_samples)} samples)", vmin, vmax)
plot_heatmap(tn_mean, tn_mean_baseline,
             f"TN (avg over {len(tn_samples)} samples)", vmin, vmax)
plot_heatmap(fp_mean, fp_mean_baseline,
             f"FP (avg over {len(fp_samples)} samples)", vmin, vmax)


In [ ]:
print("Averaged top heads (by absolute mean indirect effect):")
print_top_heads(tp_mean, f"TP (N={len(tp_samples)})")
print_top_heads(tn_mean, f"TN (N={len(tn_samples)})")
print_top_heads(fp_mean, f"FP (N={len(fp_samples)})")
